# 01 — Launch a RunPod GPU pod

This notebook runs **on your laptop** (Mac, Windows, or Linux). It uses the RunPod Python SDK to spin up a GPU pod *under your own RunPod account*, then prints a JupyterLab URL you'll open in another tab.

**Before you run this:**
1. Run `bash setup/setup.sh` (Mac/Linux) or `setup\setup.ps1` (Windows) once.
2. Copy `attendee/config.example.py` to `attendee/config.py` and fill in your RunPod API key + the storage pod URL your workshop host gave you.

## Cell 1 — Load your config
Reads `attendee/config.py`. Errors loudly if you forgot to fill in the API key or the storage pod URL.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if (REPO_ROOT / 'attendee').exists():
    pass  # we're at repo root
elif REPO_ROOT.name == 'attendee':
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError(f"Run this notebook from the repo root or attendee/. cwd={REPO_ROOT}")
sys.path.insert(0, str(REPO_ROOT / 'attendee'))
import config

assert 'REPLACE_ME' not in config.RUNPOD_API_KEY, "Edit attendee/config.py: RUNPOD_API_KEY"
assert 'REPLACE_ME' not in config.STORAGE_POD_URL, "Edit attendee/config.py: STORAGE_POD_URL"
print(f"OK. RunPod key: {config.RUNPOD_API_KEY[:8]}...   Storage URL: {config.STORAGE_POD_URL}")

## Cell 2 — Authenticate with RunPod and pick a GPU
Walks through `config.GPU_TYPE_PREFERENCE` and uses the first one RunPod has stock for.

In [ ]:
import runpod
runpod.api_key = config.RUNPOD_API_KEY

available = {gpu['id']: gpu for gpu in runpod.get_gpus()}
gpu_type = next((g for g in config.GPU_TYPE_PREFERENCE if g in available), None)
if gpu_type is None:
    raise RuntimeError(f"None of {config.GPU_TYPE_PREFERENCE} are available right now. Available: {sorted(available)}")
print(f"Using GPU: {gpu_type}")

## Cell 3 — Create the pod

The pod's `docker_args` does the bare minimum: clones the workshop repo, then runs `pod/startup.sh` from the cloned repo to do the heavy lifting (clone nanochat, download SEC data, install deps, stage data). The script logs to `/workspace/startup.log`. The training notebook waits for `Setup complete` to appear in that log.

JupyterLab itself is started by the `runpod/pytorch:*` image's default entrypoint and is up within ~30s of pod boot. The setup script keeps running in the background — you can watch it via `tail -f /workspace/startup.log` from any JupyterLab terminal.

In [ ]:
DOCKER_ARGS = (
    "bash -lc '"
    "set -e; "
    "mkdir -p /workspace && cd /workspace; "
    f"git clone {config.WORKSHOP_REPO_URL} /workspace/zero-to-llm || true; "
    "bash /workspace/zero-to-llm/pod/startup.sh || echo SETUP_FAILED >> /workspace/startup.log; "
    "sleep infinity"
    "'"
)

pod = runpod.create_pod(
    name=config.GPU_POD_NAME,
    image_name=config.GPU_POD_IMAGE,
    gpu_type_id=gpu_type,
    gpu_count=1,
    cloud_type='ALL',
    container_disk_in_gb=config.GPU_POD_DISK_GB,
    ports='8888/http,22/tcp',
    docker_args=DOCKER_ARGS,
    support_public_ip=True,
    start_ssh=True,
    env={
        'JUPYTER_TOKEN': '',
        'JUPYTER_PASSWORD': '',
        'STORAGE_POD_URL': config.STORAGE_POD_URL,
        'DATA_SCOPE': config.DEFAULT_DATA_SCOPE,
        'WORKSHOP_REPO_URL': config.WORKSHOP_REPO_URL,
    },
)
pod_id = pod['id']
jupyter_url = f'https://{pod_id}-8888.proxy.runpod.net/lab'
print(f'Pod ID    : {pod_id}')
print(f'Jupyter   : {jupyter_url}')

## Cell 4 — Wait for JupyterLab to be reachable
Polls the proxy URL every 10 s. Usually ready within 60 s of pod creation.

In [ ]:
import time, urllib.request, urllib.error
start = time.time()
for _ in range(60):
    try:
        with urllib.request.urlopen(urllib.request.Request(jupyter_url, method='HEAD'), timeout=10) as r:
            if r.status in (200, 302):
                print(f'JupyterLab is up after {int(time.time()-start)}s'); break
    except urllib.error.HTTPError as e:
        if e.code in (200, 302, 401, 403):
            print(f'JupyterLab is up after {int(time.time()-start)}s'); break
    except Exception:
        pass
    time.sleep(10)
    print(f'  ... booting ({int(time.time()-start)}s)')
else:
    print('Timed out waiting; check the RunPod console.')

## Cell 5 — Next steps

1. **Open the JupyterLab URL above in a new browser tab.**
2. In JupyterLab's file browser, navigate to: `zero-to-llm/pod/02_train_workshop.ipynb`
3. Run cells top-to-bottom. The first cell waits for the background install to finish.
4. After training, run `pod/03_chat.ipynb` to chat with your model.
5. **When you're done, terminate the pod from the RunPod console** to stop charges (the URL is printed below).

Save the pod ID printed below in case you need to come back.

In [ ]:
print(f'Pod ID            : {pod_id}')
print(f'JupyterLab URL    : {jupyter_url}')
print(f'Termination URL   : https://www.runpod.io/console/pods/{pod_id}')